# 15 - Segmentação Semântica

A **Segmentação Semântica** é a tarefa de classificar cada pixel de uma imagem em uma determinada categoria ou classe. Diferente da localização ou detecção de objetos, onde o modelo prevê caixas delimitadoras (*bounding boxes*), na segmentação o objetivo é delinear a forma exata do objeto, pixel a pixel.

Neste notebook, vamos construir um modelo FCN (Fully Convolutional Network) para realizar a segmentação semântica de dígitos espalhados em uma imagem.
Matematicamente, para uma imagem de entrada $X \in \mathbb{R}^{C \times H \times W}$, a saída do nosso modelo será um tensor de predições $Y \in \mathbb{R}^{K \times H \times W}$, onde $K$ é o número total de classes (incluindo o fundo). A função de perda utilizada será a **Entropia Cruzada (*Cross-Entropy Loss*)**, aplicada individualmente para cada pixel espacial:

$$ \mathcal{L} = -\frac{1}{H \times W} \sum_{i=1}^{H} \sum_{j=1}^{W} \sum_{k=1}^{K} y_{i,j,k} \log(\hat{y}_{i,j,k}) $$

Onde $y$ é o *ground truth* (codificado como *one-hot*) e $\hat{y}$ é a probabilidade predita pelo modelo.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import random

In [ ]:
# Definindo o dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")

### Preparação dos Dados

Vamos criar um dataset sintético utilizando os dígitos do **MNIST**. A ideia é criar imagens maiores (ex: 64x64 pixels) e colocar $N$ dígitos do MNIST (28x28 originais, mas redimensionados ou originais) em posições aleatórias nessas imagens. 
O *target* não será uma classe única, mas sim uma **máscara da mesma resolução da imagem**, onde cada pixel terá um valor correspondente à classe do dígito ali posicionado (1 a 10) ou 0 para o fundo (*background*).

In [ ]:
class MultiDigitSegmentationDataset(Dataset):
    def __init__(self, mnist_dataset, image_size=64, num_digits=3, length=1000):
        self.mnist = mnist_dataset
        self.image_size = image_size
        self.num_digits = num_digits
        self.length = length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Imagem e máscara em branco
        img = torch.zeros((1, self.image_size, self.image_size))
        mask = torch.zeros((self.image_size, self.image_size), dtype=torch.long)
        
        boxes = []
        digits_placed = 0
        
        while digits_placed < self.num_digits:
            # Seleciona um dígito aleatório do MNIST
            mnist_idx = random.randint(0, len(self.mnist) - 1)
            digit_img, label = self.mnist[mnist_idx]
            
            # O label no dataset é 0-9. Nossa máscara usará 1-10 para os dígitos e 0 para o fundo.
            class_idx = label + 1
            
            # Posição aleatória na imagem maior
            h, w = digit_img.shape[1], digit_img.shape[2]
            max_y = self.image_size - h
            max_x = self.image_size - w
            
            placed = False
            for _ in range(50): # Limite de 50 tentativas para evitar loop infinito
                y = random.randint(0, max_y)
                x = random.randint(0, max_x)
                
                # Checar colisão de caixas delimitadoras
                collision = False
                for (bx, by, bw, bh) in boxes:
                    if not (x + w <= bx or x >= bx + bw or y + h <= by or y >= by + bh):
                        collision = True
                        break
                
                if not collision:
                    boxes.append((x, y, w, h))
                    placed = True
                    break
            
            # Se não conseguiu encontrar uma posição livre, aborta este dígito (pode desenhar menos do que num_digits)
            if not placed:
                break
                
            # Copiar dígito para a imagem e criar a máscara
            # Aonde o dígito for maior que um limiar, consideramos parte da máscara
            img_crop = digit_img[0]
            mask_crop = (img_crop > 0.1).long() * class_idx
            
            # Atualiza a imagem com o dígito (somando intensidades e limitando a 1.0)
            img[0, y:y+h, x:x+w] = torch.clamp(img[0, y:y+h, x:x+w] + img_crop, 0, 1)
            
            # Atualiza a máscara: se o pixel for fundo, substitui pelo dígito. 
            current_mask_region = mask[y:y+h, x:x+w]
            mask[y:y+h, x:x+w] = torch.where(mask_crop > 0, mask_crop, current_mask_region)
            
            digits_placed += 1
            
        return img, mask

In [ ]:
# Carregar o MNIST original
transform = transforms.ToTensor()
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_dataset = MultiDigitSegmentationDataset(mnist_train, image_size=64, num_digits=3, length=3000)
test_dataset = MultiDigitSegmentationDataset(mnist_test, image_size=64, num_digits=3, length=500)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Visualizando um exemplo
imgs, masks = next(iter(train_dataloader))
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("Imagem de Entrada")
plt.imshow(imgs[0].squeeze(), cmap='gray')
plt.subplot(1, 2, 2)
plt.title("Máscara de Segmentação")
plt.imshow(masks[0], cmap='jet', vmin=0, vmax=10)
plt.show()

### Arquitetura do Modelo U-Net

Vamos utilizar uma rede inspirada na **U-Net**. A rede possui um *Encoder* que reduz as dimensões espaciais e extrai características, e um *Decoder* que aumenta as dimensões de volta ao original usando convoluções transpostas. O diferencial da U-Net são as **Skip Connections**, que concatenam os mapas de características do Encoder diretamente no Decoder, recuperando detalhes espaciais finos (como as bordas dos dígitos). A saída final será um tensor de tamanho `[Batch, 11, 64, 64]`.

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self, num_classes=11):
        super(SimpleUNet, self).__init__()
        
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2) # 64 -> 32
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2) # 32 -> 16
        
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        # Decoder
        self.upconv2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1), # 64 = 32 (up) + 32 (enc2)
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        self.upconv1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1), # 32 = 16 (up) + 16 (enc1)
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        
        self.final_conv = nn.Conv2d(16, num_classes, kernel_size=1)
        
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        # Bottleneck
        b = self.bottleneck(p2)
        
        # Decoder
        u2 = self.upconv2(b)
        cat2 = torch.cat([u2, e2], dim=1) # Skip connection 2
        d2 = self.dec2(cat2)
        
        u1 = self.upconv1(d2)
        cat1 = torch.cat([u1, e1], dim=1) # Skip connection 1
        d1 = self.dec1(cat1)
        
        out = self.final_conv(d1)
        return out

In [ ]:
model = SimpleUNet(num_classes=11).to(device)
print(model)

### Função de Perda, Otimizadores e Treinamento

A perda `nn.CrossEntropyLoss` do PyTorch foi projetada para lidar diretamente com previsões espaciais caso o formato seja correto.
Se a saída predita é `[Batch, C, H, W]` e o *ground truth* (a máscara de segmentação) for `[Batch, H, W]` (com inteiros contendo os IDs da classe de 0 até C-1), a *CrossEntropyLoss* é computada corretamente sobre todos os pixels.

In [ ]:
weights = torch.ones(11)
weights[0] = 0.1      # fundo pesa menos
weights[1:] = 2.0     # dígitos pesam mais
weights = weights.to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
num_epochs = 20
losses = []

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for imgs, masks in train_dataloader:
        imgs, masks = imgs.to(device), masks.to(device)
        
        # Forward
        outputs = model(imgs) # Saída: Batch x 11 x 64 x 64
        
        # Loss espera predições (Batch, C, d1, d2) e alvos (Batch, d1, d2)
        loss = criterion(outputs, masks)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_dataloader)
    losses.append(avg_loss)
    print(f"[{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")
    
plt.plot(losses)
plt.title("Curva de Perda no Treinamento")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.show()

### Avaliação e Visualização

Com o modelo treinado, podemos extrair as predições de segmentação na base de teste. Para determinar a classe final de cada pixel, aplicaremos a função `argmax` sobre o eixo de classes (canais) do tensor de saída.

In [ ]:
model.eval()
imgs, masks = next(iter(test_dataloader))
imgs, masks = imgs.to(device), masks.to(device)

with torch.no_grad():
    outputs = model(imgs)
    # outputs: [Batch, 11, 64, 64]
    # argmax no eixo 1 (Canais) transforma para [Batch, 64, 64] com as classes finais
    preds = torch.argmax(outputs, dim=1)

# Visualizando os resultados
num_examples = 4
fig, axes = plt.subplots(num_examples, 3, figsize=(10, 3*num_examples))

for i in range(num_examples):
    img = imgs[i].cpu().squeeze().numpy()
    mask = masks[i].cpu().numpy()
    pred = preds[i].cpu().numpy()
    
    axes[i, 0].imshow(img, cmap='gray')
    axes[i, 0].set_title("Imagem Entrada")
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(mask, cmap='jet', vmin=0, vmax=10)
    axes[i, 1].set_title("Máscara Real")
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(pred, cmap='jet', vmin=0, vmax=10)
    axes[i, 2].set_title("Máscara Predita")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## Exercícios

### Exercício 1: Data Augmentation Espacial
O modelo está aprendendo a segmentar os dígitos em posições quase normais. Modifique o código do `MultiDigitSegmentationDataset` para aplicar rotações aleatórias aos recortes do MNIST (`digit_img`) antes de colá-los na imagem maior, girando também a máscara correspondente. Isso ajudará o modelo a generalizar melhor a segmentação.

### Exercício 2: Mais ou Menos Dígitos
Altere o hiperparâmetro `num_digits` do dataset. Note que quanto mais dígitos (e mais sobreposições), mais difícil é a tarefa. Retreine o modelo com `num_digits=5` e avalie visualmente se o modelo ainda consegue diferenciar bem as sobreposições.